# TRELLIS.2 — A100 Optimized + Profiled Colab

This is the **experimental A100-focused 3D notebook**.

It keeps Microsoft's safe `low_vram=True` policy for the large 3D models, but adds a conservative **smart residency** optimization: DINOv3 stays on the GPU across the consecutive 512/1024 conditioning passes instead of being copied CPU→GPU→CPU twice.

It also profiles TRELLIS inference as separate stages and reports:
- elapsed time per sub-stage,
- peak allocated VRAM,
- peak reserved VRAM,
- model parameter footprint,
- GLB post-processing/export time.

The DINOv3 / Transformers compatibility patch is applied and verified automatically.

> Recommended runtime: **A100 40 GB or larger**.


In [ ]:
# Shared setup — fresh Colab runtime.
import os, pathlib, shutil, subprocess, time, collections

LOG_DIR = pathlib.Path("/content/engine_logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run_live(cmd, *, cwd=None, env=None, label="process"):
    cmd = [str(x) for x in cmd]
    safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in label)[:60]
    stamp = time.strftime("%Y%m%d_%H%M%S")
    log_path = LOG_DIR / f"{stamp}_{safe}.log"

    print("\n" + "=" * 78, flush=True)
    print(f"[RUN] {label}", flush=True)
    print("[CMD] " + " ".join(cmd), flush=True)
    print(f"[LOG] {log_path}", flush=True)
    print("=" * 78, flush=True)

    started = time.time()
    tail = collections.deque(maxlen=100)
    with log_path.open("w", encoding="utf-8", errors="replace") as log:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
            tail.append(line.rstrip("\n"))
        rc = proc.wait()

    elapsed = time.time() - started
    if rc != 0:
        print("\n" + "!" * 78, flush=True)
        print(f"[FAILED] {label} | exit={rc} | elapsed={elapsed/60:.1f} min", flush=True)
        print(f"[FULL LOG] {log_path}", flush=True)
        print("[LAST LOG LINES]", flush=True)
        for line in tail:
            print(line, flush=True)
        print("!" * 78, flush=True)
        raise RuntimeError(f"{label} failed with exit code {rc}. See {log_path}.")

    print(f"\n[DONE] {label} | elapsed={elapsed/60:.1f} min", flush=True)
    return log_path

print("[COLAB][1/3] Checking GPU runtime...", flush=True)
if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU runtime attached. In Colab choose Runtime → Change runtime type → GPU "
        "(A100 recommended), reconnect, then rerun this cell."
    )

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    text=True, capture_output=True, check=True,
).stdout.strip().splitlines()
gpu_name, memory_mib = [x.strip() for x in smi[0].rsplit(",", 1)]
memory_mib = int(memory_mib)
free_gib = shutil.disk_usage("/content").free / 1024**3
print(f"[COLAB][1/3] GPU: {gpu_name} | VRAM: {memory_mib/1024:.1f} GiB | Free disk: {free_gib:.1f} GiB", flush=True)
if memory_mib < 35000:
    print("[WARN] This notebook is tuned for A100 40 GB+. It will stay conservative on smaller GPUs.", flush=True)
if free_gib < 35:
    raise RuntimeError("Need at least 35 GiB free disk.")

print("[COLAB][2/3] Cloning current helpers...", flush=True)
REPO = pathlib.Path("/content/My-works")
if REPO.exists():
    shutil.rmtree(REPO)
run_live(
    ["git", "clone", "--progress", "--depth", "1",
     "https://github.com/Logan17de/My-works.git", str(REPO)],
    label="Clone My-works helpers",
)
ENGINE_ROOT = REPO / "ai-3d-animation-engines"
TOOLS_3D = ENGINE_ROOT / "3d-engine"
print("[COLAB][3/3] ✅ Helpers ready.", flush=True)


## 💾 Optional Google Drive build cache

This caches **source snapshots and compatible compiled wheels**. It does **not** store TRELLIS/DINO model weights in Drive.


In [ ]:
USE_DRIVE_BUILD_CACHE = True #@param {type:"boolean"}
DRIVE_CACHE_ROOT = "/content/drive/MyDrive/AI3D_Engine_Cache" #@param {type:"string"}

if USE_DRIVE_BUILD_CACHE:
    from google.colab import drive
    print("[CACHE][1/3] Mounting Google Drive...", flush=True)
    drive.mount("/content/drive", force_remount=False)
    cache_root = pathlib.Path(DRIVE_CACHE_ROOT)
    print("[CACHE][2/3] Preparing cache folders...", flush=True)
    for name in ("sources", "wheels", "downloads"):
        (cache_root / name).mkdir(parents=True, exist_ok=True)
    os.environ["ENGINE_CACHE_ROOT"] = str(cache_root)
    source_count = len(list((cache_root / "sources").glob("*.tar.gz")))
    wheel_count = len(list((cache_root / "wheels").rglob("*.whl")))
    print(f"[CACHE][3/3] ✅ {cache_root}", flush=True)
    print(f"[CACHE] Source snapshots: {source_count} | wheels: {wheel_count}", flush=True)
else:
    os.environ.pop("ENGINE_CACHE_ROOT", None)
    print("[CACHE] Disabled.", flush=True)


## 1. Install TRELLIS.2

The installer restores cached wheels/sources when available and applies the DINOv3 compatibility patch automatically.


In [ ]:
installer = TOOLS_3D / "install_3d.sh"
run_live(["bash", "-n", str(installer)], label="TRELLIS installer syntax check")
run_live(["bash", str(installer)], label="TRELLIS.2 installation")


## 2. Verify DINOv3 compatibility patch

Current Transformers versions wrap the DINOv3 encoder differently from the original pinned TRELLIS source. This cell verifies/applies the compatibility patch explicitly before any model load.


In [ ]:
run_live(
    [
        "/opt/conda/bin/conda", "run", "--no-capture-output", "-n", "trellis2",
        "python", str(TOOLS_3D / "patch_trellis_dinov3.py"),
    ],
    cwd="/content/TRELLIS.2",
    label="Verify DINOv3 compatibility patch",
)


## 3. Hugging Face sign-in + gated access + visible downloads

Use the same Hugging Face account that has accepted/requested access to:
- `facebook/dinov3-vitl16-pretrain-lvd1689m`
- `briaai/RMBG-2.0`

Best option: Colab **Secrets (🔑)** → add `HF_TOKEN`. A hidden prompt is used if the secret is absent.


In [ ]:
import getpass
from google.colab import userdata

HF_TOKEN = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (hidden): " ).strip()
if not HF_TOKEN:
    raise ValueError("HF_TOKEN is required.")

# Never print the token or place it on the command line.
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = "/content/huggingface"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)

hf_env = os.environ.copy()
hf_env["HF_TOKEN"] = HF_TOKEN
hf_env["HF_HOME"] = "/content/huggingface"
hf_env["HF_XET_HIGH_PERFORMANCE"] = "1"
hf_env["PYTHONUNBUFFERED"] = "1"

run_live(
    [
        "/opt/conda/bin/conda", "run", "--no-capture-output", "-n", "trellis2",
        "python", str(TOOLS_3D / "prepare_hf_models.py"),
    ],
    cwd="/content/TRELLIS.2",
    env=hf_env,
    label="HF auth + TRELLIS model pre-download",
)
print("[HF] ✅ All required runtime model files are ready in local Colab cache.", flush=True)


## 4. Upload image + choose A100 profile

### Pipeline
- `512`: much faster; useful for quick geometry tests.
- `1024_cascade`: normal high-quality TRELLIS path and the recommended default.
- `1024`: direct 1024 path; can be memory-heavy.
- `1536_cascade`: very expensive; not recommended on A100 40 GB.

### Residency
- `smart`: keeps **DINOv3 only** resident across its two consecutive conditioning passes. Large 3D models still use safe official offloading.
- `official`: reproduces Microsoft's low-VRAM transfer behavior for comparison.

For the first profile, keep `smart + 1024_cascade`.


In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one reference image.")
INPUT_IMAGE = f"/content/{next(iter(uploaded))}"

ASSET_NAME = "test_character" #@param {type:"string"}
ASSET_TYPE = "character" #@param ["object", "character", "environment", "other"]
PIPELINE_TYPE = "1024_cascade" #@param ["512", "1024_cascade", "1024", "1536_cascade"]
RESIDENCY = "smart" #@param ["smart", "official"]
SEED = 42 #@param {type:"integer"}

TARGET_AXIS = "height" #@param ["width", "height", "depth", "longest"]
TARGET_SIZE_METERS = 1.75 #@param {type:"number"}

# Balanced export defaults: cheaper than the old 1M-face / 4K test.
DECIMATION_TARGET = 500000 #@param {type:"integer"}
TEXTURE_SIZE = 2048 #@param {type:"integer"}
SKIP_PREVIEW = True #@param {type:"boolean"}

if TARGET_SIZE_METERS <= 0:
    raise ValueError("TARGET_SIZE_METERS must be positive.")
print("Input:", INPUT_IMAGE)
print(f"Pipeline={PIPELINE_TYPE} | residency={RESIDENCY} | type={ASSET_TYPE}")
print(f"Scale: {TARGET_AXIS}={TARGET_SIZE_METERS} m")
print(f"Export: {DECIMATION_TARGET:,} faces | {TEXTURE_SIZE}px | skip preview={SKIP_PREVIEW}")


## 5. Generate + profile

During sampling, TRELLIS' own progress bars are shown. After every sub-stage you'll get an **A100 PROFILE** line with elapsed time and peak VRAM.


In [ ]:
OUTPUT_DIR = "/content/trellis_a100_outputs"

cmd = [
    "/opt/conda/bin/conda", "run", "--no-capture-output", "-n", "trellis2",
    "python", str(TOOLS_3D / "run_trellis2_a100_profiled.py"),
    "--input", INPUT_IMAGE,
    "--output-dir", OUTPUT_DIR,
    "--name", ASSET_NAME,
    "--asset-type", ASSET_TYPE,
    "--pipeline-type", PIPELINE_TYPE,
    "--residency", RESIDENCY,
    "--seed", str(SEED),
    "--target-axis", TARGET_AXIS,
    "--target-size-m", str(TARGET_SIZE_METERS),
    "--decimation-target", str(DECIMATION_TARGET),
    "--texture-size", str(TEXTURE_SIZE),
]
if SKIP_PREVIEW:
    cmd.append("--skip-preview")

generation_env = os.environ.copy()
generation_env["HF_HOME"] = "/content/huggingface"
generation_env["HF_XET_HIGH_PERFORMANCE"] = "1"

run_live(
    cmd,
    cwd="/content/TRELLIS.2",
    env=generation_env,
    label="TRELLIS.2 A100 profiled generation",
)

GLB_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}.glb"
MANIFEST_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}_manifest.json"
PROFILE_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}_a100_profile.json"
PREVIEW_PATH = f"{OUTPUT_DIR}/{ASSET_NAME}_preview.mp4"

for required in (GLB_PATH, MANIFEST_PATH, PROFILE_PATH):
    if not pathlib.Path(required).is_file():
        raise RuntimeError(f"Missing expected output: {required}")
print("✅ Generation complete.")


## 6. Read the bottleneck report


In [ ]:
import json

profile = json.loads(pathlib.Path(PROFILE_PATH).read_text())
rows = sorted(profile["profile"], key=lambda r: r["seconds"], reverse=True)

print(f"GPU: {profile['gpu']['name']} | {profile['gpu']['total_vram_gib']:.1f} GiB")
print(f"Pipeline: {profile['trellis']['pipeline_type']} | residency: {profile['trellis']['residency']}")
print("\nSlowest stages:")
for i, row in enumerate(rows, 1):
    print(
        f"{i:>2}. {row['name']:<40} "
        f"{row['seconds']:>7.1f}s | "
        f"peak alloc {row['peak_allocated_gib']:>6.2f} GiB | "
        f"peak reserved {row['peak_reserved_gib']:>6.2f} GiB"
    )


In [ ]:
# Optional preview.
from IPython.display import Video, display

if pathlib.Path(PREVIEW_PATH).is_file():
    display(Video(PREVIEW_PATH, embed=True))
else:
    print("Preview skipped. GLB, manifest, and profile are ready.")


In [ ]:
# Download outputs.
from google.colab import files

files.download(GLB_PATH)
files.download(MANIFEST_PATH)
files.download(PROFILE_PATH)
